# Llama 4
Llama 4 Scout / Maverick (Meta, April 2025). There is **no paper**: the details below come from Meta's release notes, the model cards and the released reference code (`llama-models`, Hugging Face `transformers`).

**Idea:** the Llama decoder turned into a **mixture-of-experts** model that reads images and text in one token stream (**early fusion**), with a few attention changes aimed at very long contexts.

### Llama 4 vs Llama 2 vs the MoE notebook
| | Llama 2 (`llama2.ipynb`) | `moe.ipynb` (Mixtral-style) | Llama 4 (this notebook) |
|---|---|---|---|
| FFN | one SwiGLU | 8 experts, top-2, softmax weights | **1 shared expert (always on) + top-1 routed expert, sigmoid gate** |
| Gate applied to | - | expert output | expert **input** |
| Position | RoPE in every layer | RoPE in every layer | RoPE, except **every 4th layer has no position encoding (NoPE)** |
| $q, k$ before the dot product | raw | raw | **L2 / RMS-normalized** (QK-norm) in the RoPE layers |
| Long-context trick | - | - | NoPE-layer queries scaled up with position (**temperature tuning**) |
| Modalities | text | text | text + images, **early fusion** |
| Sizes | 7B - 70B | - | Scout: 16 experts, 109B total / 17B active. Maverick: 128 experts, 400B total / 17B active |

`moe.ipynb` covers routing, the sparse mixture and the load-balancing loss; `llama2.ipynb` covers GQA and the KV cache. This notebook builds only what is new: the shared + routed expert layer, QK-norm, the RoPE / NoPE interleaving and temperature tuning.

Shape flow: `idx (B, T) -> embed (B, T, D) -> [RMSNorm -> attn(RoPE + QK-norm, or NoPE) -> + -> RMSNorm -> shared expert + routed expert -> +] x L -> RMSNorm -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data and shared pieces (same as `slm.ipynb`)
Character-level tiny Shakespeare, same split and batch shape as `slm.ipynb` and `moe.ipynb`. RMSNorm, RoPE and SwiGLU are copied unchanged.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
block_size, batch_size = 128, 32   # T, B

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_cache(T, head_dim, base=10000):
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)
    angles = torch.arange(T).float()[:, None] * freqs[None]
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def ffn_hidden(dim):
    return 64 * ((8 * dim // 3 + 63) // 64)

## 2. Attention: QK-norm, RoPE / NoPE layers, temperature tuning
Three changes to the Llama 2 attention, all in the reference code's text decoder.

### Formula: QK-norm (in the RoPE layers)
$$\hat q = \frac{q}{\sqrt{\frac{1}{d}\sum_j q_j^2 + \epsilon}}, \qquad \hat k = \frac{k}{\sqrt{\frac{1}{d}\sum_j k_j^2 + \epsilon}}, \qquad \text{score} = \frac{\hat q^{\,T} \hat k}{\sqrt{d}}$$
- $q, k$: one head's query and key, dim $d$, **after** RoPE (rotation does not change the norm, so the order does not matter)
- an RMSNorm with no learned gain: every $q$ and $k$ has norm $\sqrt{d}$, so the score is $\sqrt{d} \cos(q, k)$, bounded in $[-\sqrt{d}, \sqrt{d}]$
- attention logits can no longer grow without limit as activations grow, which stabilizes training of very large models

### Formula: NoPE layers
$$\text{use\_rope}(\ell) = \big[(\ell + 1) \bmod 4 \ne 0\big], \qquad \ell = 0, 1, 2, \dots$$
- layers 3, 7, 11, ... apply **no** position encoding to $q, k$ (and no QK-norm)
- a causal model can still infer order without explicit positions; the NoPE layers attend purely by content, at any distance, which is meant to help generalize to contexts longer than those seen in training

### Formula: Attention temperature tuning (NoPE layers, inference at long context)
$$q_m \leftarrow q_m \cdot \Big(1 + \alpha \log\big(\lfloor (m + 1) / s \rfloor + 1\big)\Big)$$
- $m$: the query's position; $s = 8192$ (`floor_scale`), $\alpha = 0.1$ (`attn_scale`)
- softmax over more keys gets flatter: with thousands of candidates, attention "dilutes". Scaling the query up sharpens it again, more for later positions
- equals 1 for $m < s - 1$, so it does nothing in the first 8k tokens. In this notebook ($T = 128$) we set $s = 32$ to see it act

In [ ]:
def l2_norm(x, eps=1e-6):
    # (..., d) -> (..., d), RMS 1 along the last dim; no learned gain
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)


class Llama4Attention(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads, use_rope, floor_scale=32, attn_scale=0.1):
        super().__init__()
        self.num_heads, self.head_dim, self.use_rope = num_heads, dim // num_heads, use_rope
        self.floor_scale, self.attn_scale = floor_scale, attn_scale
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                # each (B, heads, T, hd)
        if self.use_rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
            q, k = l2_norm(q), l2_norm(k)                                   # QK-norm
        else:                                                               # NoPE layer: temperature tuning instead
            m = torch.arange(T, device=x.device).float()
            scale = torch.log(torch.floor((m + 1) / self.floor_scale) + 1) * self.attn_scale + 1   # (T,)
            q = q * scale[:, None]                                          # one factor per query position

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5              # (B, heads, T, T)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D))

q = torch.randn(4, 64) * 5
print("norm of q before / after QK-norm:", q.norm(dim=-1)[:2].tolist(), l2_norm(q).norm(dim=-1)[:2].tolist(), "| sqrt(d) =", 8.0)
m = torch.arange(128).float()
print("temperature scale at positions 0, 31, 63, 127 (s = 32):", [round((math.log(math.floor((p + 1) / 32) + 1) * 0.1 + 1), 3) for p in [0, 31, 63, 127]])
print("layers that use RoPE (L = 8):", [int((l + 1) % 4 != 0) for l in range(8)])

## 3. MoE layer: shared expert + one routed expert
Different from `moe.ipynb` in three ways.

### Formula: Llama 4 MoE
$$y = \text{FFN}_{shared}(x) + \text{FFN}_{e^*}\big(g \cdot x\big), \qquad e^* = \arg\max_i\, (W_r x)_i, \qquad g = \sigma\big((W_r x)_{e^*}\big)$$
- $x \in \mathbb{R}^D$: one token; $W_r \in \mathbb{R}^{E \times D}$: router, $E = 8$ experts here
- **shared expert**: a SwiGLU that every token passes through. It holds what all tokens need, so the routed experts can specialize
- **top-1**: each token goes to a single routed expert (Mixtral: top-2)
- **sigmoid gate** $g \in (0, 1)$: the winning expert's router logit through a sigmoid, with no normalization across experts (Mixtral: softmax over the chosen ones)
- the gate multiplies the expert's **input**, not its output
- active parameters per token: shared + 1 expert, whatever $E$ is. That is how Maverick has 400B parameters and 17B active

The reference code is inference-only and has no load-balancing term. To train from scratch we add the Switch Transformer auxiliary loss from `moe.ipynb`, computed on a softmax of the router logits; without it the router collapses onto a few experts.

In [ ]:
class Llama4MoE(nn.Module):
    # (B, T, D) -> (B, T, D), aux loss (scalar)
    def __init__(self, dim, hidden, num_experts=8):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(dim, num_experts, bias=False)                  # W_r
        self.shared_expert = SwiGLU(dim, hidden)
        self.experts = nn.ModuleList([SwiGLU(dim, hidden) for _ in range(num_experts)])

    def forward(self, x):
        B, T, D = x.shape
        x = x.reshape(-1, D)                                    # (N, D), N = B*T tokens
        logits = self.router(x)                                 # (N, E)
        top_logit, top_i = logits.max(dim=-1)                   # (N,), (N,) top-1 expert per token
        gate = torch.sigmoid(top_logit)                         # (N,) in (0, 1)

        out = self.shared_expert(x)                             # every token
        for e, expert in enumerate(self.experts):
            idx = (top_i == e).nonzero(as_tuple=True)[0]        # tokens routed to expert e
            if idx.numel() == 0:
                continue
            out = out.index_add(0, idx, expert(x[idx] * gate[idx, None]))   # gate scales the INPUT

        f = F.one_hot(top_i, self.num_experts).float().mean(0)  # (E,) share of tokens per expert
        P = logits.softmax(dim=-1).mean(0)                      # (E,) mean router probability
        aux = self.num_experts * (f * P).sum()                  # = 1 when routing is uniform
        self.last_load = f.detach()
        return out.reshape(B, T, D), aux

moe = Llama4MoE(64, 192)
y, aux = moe(torch.randn(2, 10, 64))
print(y.shape, "| aux:", round(aux.item(), 3), "| load:", [round(v, 2) for v in moe.last_load.tolist()])

## 4. The decoder
Layer $\ell$ uses RoPE + QK-norm unless $(\ell + 1) \bmod 4 = 0$. Every layer has the MoE feed-forward, as in **Scout**. (Maverick alternates: dense SwiGLU in odd layers, MoE with 128 experts in the others.)

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, num_experts, use_rope):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), Llama4Attention(dim, num_heads, use_rope)
        self.norm2, self.moe = RMSNorm(dim), Llama4MoE(dim, ffn_hidden(dim), num_experts)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        y, aux = self.moe(self.norm2(x))
        return x + y, aux


class Llama4(nn.Module):
    # idx (B, T) -> logits (B, T, V), aux (scalar)
    def __init__(self, vocab_size, block_size, dim=256, depth=4, num_heads=4, num_experts=8, nope_interval=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads, num_experts, use_rope=(l + 1) % nope_interval != 0) for l in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.normal_(self.embed.weight, std=0.02)
        cos, sin = rope_cache(block_size, dim // num_heads)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.embed(idx)
        aux = 0.0
        for block in self.blocks:
            x, a = block(x, self.cos[:T], self.sin[:T])
            aux = aux + a
        return self.head(self.norm(x)), aux

model = Llama4(vocab_size, block_size).to(device)
total = sum(p.numel() for p in model.parameters())
per_expert = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
inactive = len(model.blocks) * (8 - 1) * per_expert          # routed experts a token does not use
print(f"total params: {total / 1e6:.2f}M | active per token: {(total - inactive) / 1e6:.2f}M")
print("RoPE per layer:", [int(b.attn.use_rope) for b in model.blocks])
print(model(torch.zeros(2, 10, dtype=torch.long, device=device))[0].shape)

## 5. Training
Same loop and total loss as `moe.ipynb`: cross-entropy plus $\alpha = 0.01$ times the summed load-balancing loss.

In [ ]:
max_iters = 3000
eval_every = 500
aux_weight = 0.01
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def loss_fn(model, x, y):
    logits, aux = model(x)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1)), aux

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s))[0].item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    ce, aux = loss_fn(model, *get_batch("train"))
    loss = ce + aux_weight * aux
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 6. Expert usage and generation
Share of tokens each routed expert received on one validation batch, per layer (uniform = $1/8$), then a sample.

In [ ]:
model.eval()
with torch.no_grad():
    model(get_batch("val")[0])
load = torch.stack([b.moe.last_load for b in model.blocks]).cpu()     # (L, E)
plt.figure(figsize=(6, 3.5))
plt.imshow(load, cmap="viridis", vmin=0); plt.colorbar(label="share of tokens")
plt.xlabel("expert"); plt.ylabel("layer"); plt.show()

@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)
    for _ in range(n):
        logits, _ = model(idx[:, -block_size:])
        idx = torch.cat([idx, torch.multinomial(F.softmax(logits[:, -1] / temperature, dim=-1), 1)], dim=1)
    return decode(idx[0].tolist())

print(generate(model))

## Notes
- **Early fusion** is the other half of Llama 4 and is not built here. Images go through a vision encoder (based on MetaCLIP), are projected to the text embedding size, and their patch tokens are placed **in the same sequence** as the text tokens from the first layer on, with all weights trained jointly on text, image and video. `vlm.ipynb` builds exactly this "image tokens in the token stream" design at small scale; Llama 3.2-Vision instead used separate cross-attention layers.
- Sizes (from the model cards): **Scout** 16 experts, ~109B total / 17B active, 10M-token context; **Maverick** 128 experts, ~400B total / 17B active, 1M-token context. Both run one routed expert plus the shared expert per token.
- A shared expert next to routed ones is also used by DeepSeek-V2/V3; it removes the need for every routed expert to relearn common features.
- QK-norm comes from the vision literature (ViT-22B, Swin v2) where unbounded attention logits caused divergence at scale. In those versions a learned temperature replaces $1 / \sqrt{d}$; here the fixed scale is kept.
- The NoPE + temperature-tuning combination follows work on length generalization (Kazemnejad et al., 2023, on NoPE; scalable-softmax-style scaling for long contexts). Since there is no paper, Meta's reasons are inferred from the code and the release notes rather than documented.
- Not shown: GQA and the KV cache (`llama2.ipynb`), chunked attention in the vision adapter, and the training recipe (MetaP, distillation from the unreleased "Behemoth" teacher).